# Qwen3.8-9B Recurrent-Loop (`K=1` vs `K=6`) — Fast Precompiled Colab T4 Demo

Run **Qwen3.8-9B** with **1 to 6 internal reasoning loops** (`152` virtual layers at `K=6` reusing the same `32` quantized blocks) on a free Google Colab **T4 GPU (16 GB VRAM)** — **no compilation needed** (downloads prebuilt Linux CUDA binaries in ~3 seconds).

- **Engine**: [`CympUser/llama.cpp-qwen-loop`](https://github.com/CympUser/llama.cpp-qwen-loop)
- **Base Model**: [`mradermacher/Qwen3.8-9B-Distill-uncensored-heretic-GGUF`](https://huggingface.co/mradermacher/Qwen3.8-9B-Distill-uncensored-heretic-GGUF) (`Q4_K_S`)
- **Loop-6 Adapter (`45.27 MB`)**: [`Vnswz/Qwen3.8-9B-Loop6-Adapter`](https://huggingface.co/Vnswz/Qwen3.8-9B-Loop6-Adapter)

> Make sure your Colab runtime is set to **Runtime → Change runtime type → T4 GPU**.

In [ ]:
# 1. Check GPU, ensure CUDA 12 runtime libs, & download precompiled Linux CUDA binary (~5s)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!mkdir -p qwen-loop/models
%cd qwen-loop

!wget -q --show-progress -O llama-linux-cuda.tar.gz \
  https://github.com/CympUser/llama.cpp-qwen-loop/releases/download/v1.0.0/llama-cpp-qwen-loop-v1.0.0-linux-cuda-x64.tar.gz
!tar -xzf llama-linux-cuda.tar.gz && chmod +x llama-cli llama-server

# Locate libcudart.so.12 & libcublas.so.12 (install lightweight pip wheels if Colab lacks CUDA 12 libs)
import glob, os, subprocess, sys

def find_cuda12_lib_dirs():
    dirs = set()
    patterns = [
        "/usr/local/cuda-12*/lib64/libcudart.so.12*",
        "/usr/local/cuda-12*/lib64/libcublas.so.12*",
        "/usr/local/cuda/lib64/libcudart.so.12*",
        "/usr/local/cuda/lib64/libcublas.so.12*",
        "/usr/local/lib/python*/dist-packages/nvidia/*/lib/libcudart.so.12*",
        "/usr/local/lib/python*/dist-packages/nvidia/*/lib/libcublas.so.12*",
        "/usr/lib/x86_64-linux-gnu/libcudart.so.12*",
        "/usr/lib/x86_64-linux-gnu/libcublas.so.12*",
    ]
    for pat in patterns:
        for p in glob.glob(pat):
            dirs.add(os.path.dirname(p))
    return sorted(dirs)

lib_dirs = find_cuda12_lib_dirs()
has_cudart = any(os.path.exists(os.path.join(d, "libcudart.so.12")) for d in lib_dirs)
has_cublas = any(os.path.exists(os.path.join(d, "libcublas.so.12")) for d in lib_dirs)
if not (has_cudart and has_cublas):
    print("Installing lightweight CUDA 12 runtime libs (nvidia-cuda-runtime-cu12, nvidia-cublas-cu12)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nvidia-cuda-runtime-cu12", "nvidia-cublas-cu12"])
    lib_dirs = find_cuda12_lib_dirs()

os.environ["LD_LIBRARY_PATH"] = ":".join(lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")])
print("CUDA 12 lib dirs:", lib_dirs)
!./llama-cli --version

In [ ]:
# 2. Download Base GGUF (5.23 GB) and Loop-6 Adapter (45 MB)
!wget -q --show-progress -O models/Qwen3.8-9B-Distill-uncensored-heretic.Q4_K_S.gguf \
  https://huggingface.co/mradermacher/Qwen3.8-9B-Distill-uncensored-heretic-GGUF/resolve/main/Qwen3.8-9B-Distill-uncensored-heretic.Q4_K_S.gguf
!wget -q --show-progress -O models/qwen3.8-9b-heretic-loop6-lora.gguf \
  https://huggingface.co/Vnswz/Qwen3.8-9B-Loop6-Adapter/resolve/main/qwen3.8-9b-heretic-loop6-lora.gguf
!ls -lh models/

In [ ]:
# 3. Helper to run the exact same prompt at any loop depth K in 1..6
import os, subprocess

def run_qwen_loop(prompt: str, num_loops: int = 6, max_tokens: int = 512) -> str:
    cmd = [
        "./llama-cli",
        "-m", "models/Qwen3.8-9B-Distill-uncensored-heretic.Q4_K_S.gguf",
        "--lora", "models/qwen3.8-9b-heretic-loop6-lora.gguf",
        "--override-kv", f"qwen35.num_loops=int:{num_loops},qwen35.recurrent_kv_cache_mode=str:per_loop",
        "-c", "8192", "-ngl", "999", "--fit", "off",
        "--cache-type-k", "q8_0", "--cache-type-v", "q8_0",
        "--temp", "0.0", "-n", str(max_tokens),
        "-st", "--simple-io",
        "-p", prompt,
    ]
    res = subprocess.run(cmd, capture_output=True, text=True, env=os.environ.copy())
    if res.returncode != 0:
        print(res.stderr)
    return res.stdout

In [ ]:
# 4. Side-by-side comparison: 4-Item Deductive Ordering (K=1 vs K=6)
logic_prompt = (
    "Four servers (Alpha, Beta, Gamma, Delta) are ranked from fastest to slowest. "
    "Clues: (1) Beta is faster than Gamma. (2) Alpha is slower than Gamma. "
    "(3) Delta is faster than Beta. "
    "Reply ONLY with the final order from fastest to slowest using '>' (e.g., A>B>C>D)."
)

print("=== K=1 (Base single-pass) ===")
print(run_qwen_loop(logic_prompt, num_loops=1, max_tokens=384))

print("=== K=6 (6 Internal Reasoning Loops) ===")
print(run_qwen_loop(logic_prompt, num_loops=6, max_tokens=384))

In [ ]:
# 5. Try your own prompt! Change NUM_LOOPS between 1 and 6
MY_PROMPT = "Explain in 3 concise bullet points why recurrent-depth transformers save VRAM compared to deeper models."
NUM_LOOPS = 6  # Try 1, 2, 4, or 6

print(run_qwen_loop(MY_PROMPT, num_loops=NUM_LOOPS, max_tokens=512))